In [1]:
from dotenv import load_dotenv
import anthropic, json, pandas as pd
from pathlib import Path
import os
import re

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [4]:
env_path = root / ".env"
print(".env existe:", env_path.exists())
load_dotenv(env_path, override=True)
assert os.environ.get("ANTHROPIC_API_KEY"), f"No hay ANTHROPIC_API_KEY en {env_path}"

client = anthropic.Anthropic()  # lee ANTHROPIC_API_KEY del entorno

.env existe: True


In [ ]:
# df = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding="utf-8")

In [ ]:
# df[(df.asignatura == "Mates CCSS") & (df.tema == "funciones ccss" )]

In [5]:
def parse_response(texto):
    texto = re.sub(r"```(?:json)?", "", texto).strip()
    dec = json.JSONDecoder()
    objs, i = [], 0
    while True:
        i = texto.find("{", i)
        if i == -1:
            break
        try:
            obj, fin = dec.raw_decode(texto, i)
            if isinstance(obj, dict):
                objs.append(obj)
            i = fin
        except json.JSONDecodeError:
            i += 1

    if not objs:
        raise ValueError(f"Sin JSON en la respuesta: {texto[:150]!r}")

    tipos, obs = [], []
    for o in objs:
        for t in str(o.get("tipo_ejercicio", "")).split("|"):
            t = t.strip()
            if t and t not in tipos:
                tipos.append(t)
        for x in str(o.get("observaciones", "") or "").split(";"):
            x = x.strip()
            if x and x not in obs:
                obs.append(x)

    orden = {"baja": 0, "media": 1, "alta": 2}
    conf = min((o.get("confianza", "baja") for o in objs),
               key=lambda c: orden.get(c, 0))
    return {"tipo_ejercicio": "|".join(tipos),
            "observaciones": "; ".join(obs),
            "confianza": conf}

In [6]:
MODELO_CLASIFICADOR = "claude-haiku-5-5"

In [7]:
def classify(enunciado, system_md, modelo=MODELO_CLASIFICADOR):
    resp = client.messages.create(
        model=modelo,
        max_tokens=1000,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )
    texto = resp.content[0].text
    try:
        return parse_response(texto)
    except Exception as e:
        # Para depurar: guarda la respuesta cruda junto al error
        raise ValueError(f"{e} | RAW: {texto[:300]!r}") from e

In [12]:
def classify(enunciado, system_md, modelo=MODELO_CLASIFICADOR):
    resp = client.messages.create(
        model=modelo,
        max_tokens=2000,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )
    # Quedarse solo con los bloques de texto (ignora los de razonamiento)
    texto = "".join(b.text for b in resp.content if b.type == "text")

    if resp.stop_reason == "max_tokens":
        raise ValueError(f"Respuesta cortada por max_tokens | RAW: {texto[:300]!r}")
    try:
        return parse_response(texto)
    except Exception as e:
        raise ValueError(f"{e} | RAW: {texto[:300]!r}") from e

In [8]:
def classify_statements(asignatura, tema, prompt_path, csv_enun = root / "csv/ejercicios_clasificados.csv" ):

    # cargar dataframes
    df_enun = pd.read_csv(csv_enun, encoding = "utf-8")
    # máscara para filtrar asignatura y tema
    mask = (df_enun.asignatura == asignatura) & (df_enun.tema == tema)
    df_to_be_classified = df_enun.loc[mask, ["id", "enunciado"]]

    system_md = Path(prompt_path).read_text(encoding="utf-8")

    # borrar el archivo si existe clasificación previa
    output_file = root / "classify_using_api/response.jsonl"
    
    try:
        if output_file.exists():
            output_file.unlink()
    except Exception as e:
        print("Archivo response.jsonl does not exist.")

    # clasificar las líneas del dataframe para clasificar
    for index, row in df_to_be_classified.iterrows():
    
        try:
            r = classify(row.enunciado, system_md)
            r["id"] = row.id
        except Exception as e:
            r = {"id": row.id, "error": str(e)}
        # important!! use write mode = "a" to add
        # classification of each row to results file
        with output_file.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # leer los ejercicios clasificados desde el json
    df_classified = pd.read_json(root / "classify_using_api/response.jsonl", lines = True)

    # añadir campo de asignatura y tema
    df_classified["asignatura"] = asignatura
    df_classified["tema"] = tema

    return df_classified

In [20]:
df = pd.read_csv(root / "classify_using_api/df_equilibrio_quimico.csv", encoding="utf-8").head().to_csv(root / "classify_using_api/df_equilibrio_quimico_sample.csv", index = False)

In [29]:
df_classified = classify_statements(asignatura="Química", tema = "equilibrio químico",
                                    prompt_path = root / "classify_using_api/equilibrio_quimico.md",
                                     csv_enun= root / "classify_using_api/df_equilibrio_quimico_sample.csv")

In [30]:
df_classified.head()

,tipo_ejercicio,observaciones,confianza,id,asignatura,tema
0,Equilibrio heterogéneo|Cálculo de Kc y Kp|Ley ...,,alta,quimica_equilibrio_quimico_2026_junio_ejercici...,Química,equilibrio químico
1,Cálculo de Kc y Kp|Grado de disociación (cálculo),,alta,quimica_equilibrio_quimico_2025_julio_ejercici...,Química,equilibrio químico
2,Equilibrio heterogéneo|Cálculo de Kc y Kp,,alta,quimica_equilibrio_quimico_2025_junio_ejercici...,Química,equilibrio químico
3,Principio de LeChatelier|Equilibrio heterogéneo,,alta,quimica_equilibrio_quimico_2024_julio_ejercici...,Química,equilibrio químico
4,Cálculo de Kc y Kp,,alta,quimica_equilibrio_quimico_2024_julio_ejercici...,Química,equilibrio químico


In [31]:
df_classified.tipo_ejercicio = df_classified.tipo_ejercicio.str.split("|")

In [33]:
df_classified = df_classified.explode("tipo_ejercicio", ignore_index=True)

In [36]:
df_classified

,tipo_ejercicio,observaciones,confianza,id,asignatura,tema
0,Equilibrio heterogéneo,,alta,2026_junio_ejercicio_3a,Química,equilibrio químico
1,Cálculo de Kc y Kp,,alta,2026_junio_ejercicio_3a,Química,equilibrio químico
2,Ley de Dalton,,alta,2026_junio_ejercicio_3a,Química,equilibrio químico
3,Cálculo de Kc y Kp,,alta,2025_julio_ejercicio_3a,Química,equilibrio químico
4,Grado de disociación (cálculo),,alta,2025_julio_ejercicio_3a,Química,equilibrio químico
5,Equilibrio heterogéneo,,alta,2025_junio_ejercicio_3a,Química,equilibrio químico
6,Cálculo de Kc y Kp,,alta,2025_junio_ejercicio_3a,Química,equilibrio químico
7,Principio de LeChatelier,,alta,2024_julio_ejercicio_b2,Química,equilibrio químico
8,Equilibrio heterogéneo,,alta,2024_julio_ejercicio_b2,Química,equilibrio químico
9,Cálculo de Kc y Kp,,alta,2024_julio_ejercicio_c1,Química,equilibrio químico


In [35]:
df_classified.id = df_classified.id.str.replace("quimica_equilibrio_quimico_", "")

In [ ]:
df_classified.to_csv("csv/ejercicios_clasificados_funciones.csv", index = False, encoding="utf-8")